# QFAC Roadmap (Oct 2025 → Mar 2026)

> Goal: By the end of this plan, demo live a Memory OS running compression-aware retrieval with observer-dependent views, switchable classical → hybrid → quantum modes, and deterministic shadow logs exposed on Hub metrics. Builds on current engines, health schema, and QFAC integration points (store/retrieve/compress/status).

## Demo outcomes
- Live Hub dashboard shows real-time KPIs: coherence, drift, compression ratio, recall parity.
- Observer-dependent views visibly change rankings (Observer A vs B).
- Mode toggles: Classical vs Hybrid vs Quantum, with deterministic shadow logs and health gates.
- Aligns with documented memory health schema and Hub endpoints.

## Timeline (6 months)
- Phase 0: Oct 1–15 — Lock the foundations
- Phase 1: Oct 16–Nov 15 — Classical QFAC baseline + retrieval
- Phase 2: Nov 16–Dec 15 — Observer-effect validator + shadow logs
- Phase 3: Dec 16–Jan 31 — Quantum Bridge hardening (sim → hybrid)
- Phase 4: Feb 1–Feb 28 — Fractal signatures & background GC
- Phase 5: Mar 1–Mar 31 — Reviewer demo & artifacts

## Phase 0 (Oct 1–15): Lock the foundations
### Goals
- Freeze canonical contracts and health metrics; wire to Hub /metrics with burn-in graphs (coherence_score, contradiction_count, drift_percent, compression_ratio).
- Keep adapters stable for legacy callers (typed MemoryRecallResult internally; legacy list-of-dicts at the edge).
### Deliverables
- ‘Contracts & Metrics’ doc + CI check.
- Hub panel: Memory Health & Compression.
### Exit criteria
- Metrics publish reliably for 7 days; no API shape changes (protected by adapter policy).

## Phase 1 (Oct 16–Nov 15): Classical QFAC baseline + retrieval
### Goals
- Implement compression-aware retrieval pipeline (fast codes → partial decode → full materialization) via QFAC APIs (store_memory, retrieve_memory, compress_all_eligible).
- Adaptive eligibility policy: start from defaults (age > 24h, access_count < 3) and gate by drift/coherence from health report.
- Stand up A/B recall benchmark; emit series to Hub for parity tracking.
### Deliverables
- qfac/retrieval.py (staged decode/rerank).
- Policy: configs/qfac.yaml (thresholds & budgets).
- A/B parity export to /metrics.
### Exit criteria
- Recall@K parity within 1% vs. uncompressed (dev set).
- P95 retrieval < 35ms (hot path).
- Compression improves store footprint ≥3× for text. (Metrics visible on Hub.)

## Phase 2 (Nov 16–Dec 15): Observer-effect validator + shadow logs
### Goals
- Add Observer-Effect Validator during compression: fidelity probe + rollback on threshold failure (continuous monitoring and validation).
- Implement Quantum Shadow Logs across encode/retrieve/interference/ECC: circuit fingerprint, backend id, redacted inputs, fidelity estimate; surface via /api/memory/status & /metrics.
### Deliverables
- qfac/validator.py, qfac/shadow_log.py.
- Hub panel: Shadow Fidelity & Rollbacks.
### Exit criteria
- ≥99% of compression events produce a shadow record.
- Validator blocks low-fidelity writes automatically; green gate required for ‘quantum’ mode.

## Phase 3 (Dec 16–Jan 31): Quantum Bridge hardening (sim → hybrid)
### Goals
- Standardize Quantum Bridge API & dataclasses (encode, retrieve, interference, ECC, stats) and enforce deterministic simulation fallback.
- Ship two circuit templates (e.g., VQE/EfficientSU2 encoder and a quantum autoencoder) and expose noise-mitigation knobs (ZNE, shots).
- Ensure deterministic shadow states are stored for all quantum/hybrid runs.
### Deliverables
- quantum/bridge.py (final), quantum/templates/{vqe_su2, quantum_autoencoder}.py.
- Config: AETHERRA_QFAC_MODE gating (classical|hybrid|quantum) wired to validator & shadow health.
### Exit criteria
- Hybrid mode on by default in demo; quantum unlocks only when last N shadow runs are green.

## Phase 4 (Feb 1–Feb 28): Fractal signatures & background GC
### Goals
- Add FractalSignature (multi-scale motif dictionary) to each QFAC node; run a Fractal GC daemon that merges motifs and rewrites cold nodes within a budget; integrate with fractal/episodic subsystems and link graph APIs.
- Expose temporal replay quality metrics (episode reconstruction fidelity, coverage) in /metrics and via orchestrator narratives/maintenance.
### Deliverables
- qfac/fractal_sig.py, qfac/rewrite_daemon.py.
- Metrics: motif reuse, rewrite budget, episode fidelity.
### Exit criteria
- Cold-tier compression ≥5× with semantic similarity ≥0.97 and stable episodic reconstruction.

## Phase 5 (Mar 1–Mar 31): Reviewer demo & artifacts
### Goals
- Live Hub with toggles: Observer A vs B, Classical vs Hybrid vs Quantum, real-time coherence/recall/compression charts; narratives & episode replays update live.
- Publish an evaluation harness (scripts + JSONL results) and a reproducible notebook.
### Deliverables
- ‘QFAC Demo Playbook’ + video, dataset card, results pack.
- Short paper preprint (methods, KPIs, ablations).
### Exit criteria
- Demo runs end-to-end in <5 minutes; one-click script reproduces metrics.

## KPIs & success gates (wired to Hub)
- Health: coherence_score ≥ 0.95; contradiction_count non-increasing WoW; drift_percent < 8%.
- Retrieval: Recall@K parity ≥ 0.99 vs uncompressed on dev set (A/B harness).
- Performance: P50/P95 retrieval < 12ms / < 35ms on hot tier.
- Compression: ≥3× (text fragments), ≥5× (fractal/episodic bundles) at semantic sim ≥ 0.97.
- Quantum hygiene: 100% of hybrid/quantum runs produce a shadow record; quantum mode gated by last-N validator pass rate.

## Demo storyboards
- Observer A vs B: Same query, different priors → different top-K & narratives; validator stamps fidelity; logs show view transformation.
- Classical vs Hybrid: Toggle mode; same recall, smaller footprint, stable coherence; shadow logs appear for hybrid runs (green).
- Episodic replay: Pick an event; reconstruct episode from compressed fractal motifs; show fidelity and coverage.

## Engineering worklist (drop-in file map)
- aetherra_core/memory/qfac/: retrieval.py (staged decode), validator.py, shadow_log.py, fractal_sig.py, rewrite_daemon.py, metrics.py
- aetherra_core/quantum/: bridge.py, templates/vqe_su2.py, templates/quantum_autoencoder.py
- tools/: ab_recall_benchmark.py (exists) + compression_eval.py (new) to dump JSONL runs
- configs/qfac.yaml: thresholds, budgets, gates

## Validation & QA
- Continuous validation (observer-effect): real-time fidelity checks; rollback on fail; adaptive parameter tweaks; integrated with Pulse/health.
- Dark-launch: dual-write raw + compressed for a subset; nightly parity check.
- Repro seeds: determinism flags + seeded embeddings (deterministic modes).

## Risk & mitigation
- Quantum backend availability → Always simulate; log QuantumBridgeUnavailable; route to classical.
- Recall degradation → A/B gates block rollout; eligibility policy tied to drift/coherence, not time alone.
- Complexity creep → Keep typed contracts + adapters for legacy code to avoid cascading changes.

## References
- Memory docs: docs/AETHERRA_MEMORY_SYSTEM.md
- QFAC file index: docs/QFAC_FILE_INDEX.md
- Config: configs/qfac.yaml
- Hub metrics endpoints: see project overview /metrics section

## Phase 1 status update (2025-09-20)

Completed in this iteration:
- Metrics schema stabilization in the hub exporter for QFAC policy and compression snapshot (always-on with safe defaults)
- Retrieval parity metrics and threshold-drop counters (with exporter integration)
- Config gauges for retrieval threshold and parity enablement
- Behavior tests covering threshold filtering and parity toggle; schema tests for all new series

Operator controls:
- AETHERRA_QFAC_RETRIEVAL_THRESHOLD: float threshold to drop low scores during retrieval (default 0.0)
- AETHERRA_QFAC_RETRIEVAL_PARITY: 1 to enable, 0 to disable parity counting (default 1)

Next steps:
- Optional: admin endpoint to reset/show parity counters and display config
- Optional: parity ratio gauge (e.g., top1_match/total) and per-k breakdowns
- Phase 2: index-level upgrades (IVF-PQ path parity), fidelity-aware reranking, and evaluation harness expansion

### QFAC 2.5 — Targeted Upgrades: Admin + Observability

We landed two small, high-impact improvements and exposed them via both CLI and Hub HTTP endpoints:

- Admin CLI (tools/qfac_admin.py)
  - --show: Prints JSON snapshot with retrieval policy and parity counters
  - --reset: Resets retrieval parity counters (safe by default)
  - Defaults to safe/quiet mode: does not instantiate a live QFAC instance unless AETHERRA_QFAC_ADMIN_ENABLE_LIVE=1
  - Output is JSON-only; incidental prints are suppressed

- Prometheus exporter: aetherra_qfac_retrieval_parity_ratio (gauge)
  - Ratio of top1 matches to total comparisons; exports 0 when total=0 for stability
  - Schema enforced in tests; always present (default 0) to maintain discoverability

- Hub HTTP endpoints (thin wrappers around CLI functions)
  - GET /api/qfac/admin/show → same JSON as CLI --show
  - POST /api/qfac/admin/reset → same JSON as CLI --reset
  - Optional control token guard: set AETHERRA_HUB_CONTROL_TOKEN
    - Provide Authorization: Bearer <token> or X-Aetherra-Control-Token: <token>
  - Inherits safe/quiet defaults from the CLI. To enable live QFAC inspection/mutation, set AETHERRA_QFAC_ADMIN_ENABLE_LIVE=1 for the Hub process.

Operator quickstart (local):
- Show snapshot (HTTP): GET http://127.0.0.1:3001/api/qfac/admin/show
- Reset counters (HTTP): POST http://127.0.0.1:3001/api/qfac/admin/reset
- CLI equivalents:
  - python tools/qfac_admin.py --show
  - python tools/qfac_admin.py --reset

Notes:
- With safe defaults, show returns available=false and reset returns ok=false (qfac unavailable). Set AETHERRA_QFAC_ADMIN_ENABLE_LIVE=1 to operate on a live instance.
- If AETHERRA_HUB_CONTROL_TOKEN is set, you must include the token header to access these endpoints.
